In [0]:
# Load the table into a DataFrame
df = spark.read.table("projects.healthcare.bronze_patient")

# Display the first 10 rows in Databricks
display(df)

In [0]:
from pyspark.sql.functions import col, round

# 1. Round Billing_Amount to 2 decimal places
df_clean = df.withColumn(
    "Billing_Amount",
    round(col("Billing_Amount"), 2)
)

# 2. Write df_clean to Silver Delta Table
df_clean.write.mode("overwrite").saveAsTable("projects.healthcare.silver_patient")

#3. Select the data
spark.read.table("projects.healthcare.silver_patient").display()



**1.Patients withs similar Blood Group in case of emergency**

In [0]:
df = spark.read.table("projects.healthcare.silver_patient");
df.filter(
    col("Blood_Type") == "O-"
).select(
    col("Name"), 
    col("Insurance_Provider")
).show()

In [0]:

filtered_df = df.filter(
    col("Blood_Type") == "O-"
)

display(filtered_df)



**2. Patients profile with details like Last Doctor, medication, blood group etc.**

In [0]:
from pyspark.sql.functions import count

df.groupBy("Name").agg(
    count("Doctor").alias("Count")).show()

In [0]:
df.filter(
    col("Name") == "Christina Williams"
).orderBy(
    col("Date_of_Admission").desc()
).show(1)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# Define window specification
window_spec = Window.partitionBy("Name") \
                     .orderBy(F.col("Date_of_Admission").desc())

# Add row number
df_ranked = df.withColumn(
    "rn",
    F.row_number().over(window_spec)
)

# Get only the latest visit for each patient
df_latest = df_ranked.filter(F.col("rn") == 1) \
                     .drop("rn")

display(df_latest)

In [0]:
profile_df = df.select(
    col("Name"), 
    col("Doctor"),
    col("Blood_Type"),
    col("Medication")
)

display(profile_df)

**3. Search the name of the patient from database**

In [0]:
search_df = df.filter(
    col("Name") == "Frank Mccormick"
)

search_df.display()

In [0]:
from pyspark.sql import functions as f
df.filter(f.col("Name").contains("John")).show()

**4.Most common Medical Condition**

In [0]:
df.groupBy(f.col("Medical_Condition")).count().orderBy(f.desc("count")).limit(1).show()

**5.Which ages are effected the most?**

In [0]:
df.groupBy(f.col("Age")).count().orderBy(f.desc("count")).limit(1).show()

**6.Diabetes effect which blood type most?**

In [0]:
df.filter(f.col("Medical_Condition") == "Diabetes").groupBy(f.col("Blood_Type")).count().orderBy(f.desc("count")).limit(1).show()


**7.Effect of Medical Condition based on gender?**

In [0]:
df.groupBy(f.col("Gender")).count().orderBy(f.desc("count")).limit(1).show()


**Create a Bar Graph - Show the Billing amount based on Medical condition**

In [0]:
from matplotlib import pyplot as plt
import numpy as np

dfData=df.groupBy(f.col("medical_condition")).agg(f.sum("billing_amount").alias("billing_amount")).toPandas()
plt.bar(dfData["medical_condition"], dfData["billing_amount"], color="red")
plt.show()

In [0]:
dfgraphData1 = df.select(
    f.col("Medical_Condition"),
    f.col("Billing_Amount")
)
display(dfgraphData1)

Databricks visualization. Run in Databricks to view.

In [0]:
dfSilverTable = spark.read.table("projects.healthcare.silver_patient")
display(dfSilverTable)

In [0]:
dfSilverTable.write.mode("overwrite").saveAsTable("projects.healthcare.gold_patient")
dfGoldTable = spark.read.table("projects.healthcare.gold_patient")
display(dfGoldTable)

